In [ ]:
import os, re, json, gzip
import pandas as pd

EVENT_XLSX   = r"2024-5 Event Module Room.xlsx"
ROOMS_XLSX   = r"Rooms and Room Types.xlsx"
STUDENT_XLSX = r"2024-5 Student Programme Module Event.xlsx" 
DPT_XLSX     = r"2024-5 DPT Data.xlsx"
PC_XLSX      = r"Programme-Course.xlsx"

# Output directory
OUTDIR = r"processed_timetabling"
os.makedirs(OUTDIR, exist_ok=True)

EVENT_SHEET   = "2024-5 Event Module Room"
ROOMS_SHEET   = "Room"
STUDENT_SHEET = "2024-5 Student Programme Module"
DPT_SHEET     = "Sheet1"
PC_SHEET      = "CourseModule"

In [2]:
DATA_DIR = "data"

STUDENT_XLSX = os.path.join(DATA_DIR, "2024-5 Student Programme Module Event.xlsx")

In [ ]:
def xlsx_sheet_to_csv_stream(
    xlsx_path: str,
    sheet_name: str,
    out_csv_path: str,
    keep_cols=None,           
):
    from openpyxl import load_workbook
    import csv

    wb = load_workbook(xlsx_path, read_only=True, data_only=True)
    if sheet_name not in wb.sheetnames:
        raise ValueError(f"Sheet not found: {sheet_name}. Available={wb.sheetnames}")

    ws = wb[sheet_name]
    rows = ws.iter_rows(values_only=True)

    header = next(rows)
    header = [str(h).strip() if h is not None else "" for h in header]

    if keep_cols is not None:
        keep = {c.strip() for c in keep_cols}
        idxs = [i for i, h in enumerate(header) if h in keep]
        kept_header = [header[i] for i in idxs]
    else:
        idxs = list(range(len(header)))
        kept_header = header

    os.makedirs(os.path.dirname(out_csv_path) or ".", exist_ok=True)
    with open(out_csv_path, "w", encoding="utf-8", newline="") as f:
        w = csv.writer(f)
        w.writerow(kept_header)
        for r in rows:
            w.writerow([r[i] for i in idxs])

    wb.close()
    print("Saved:", out_csv_path)

STUDENT_CSV = "student_programme_module_event.csv"

KEEP = ["AnonID", "Programme Code-Year", "Event ID", "Semester", "Course ID", "Department", "Programme", "Course Name"]

xlsx_sheet_to_csv_stream(STUDENT_XLSX, STUDENT_SHEET, STUDENT_CSV, keep_cols=KEEP)

Saved: student_programme_module_event.csv


In [4]:
import os, glob

DATA_DIR = "data"

def pick_one(patterns):
    hits = []
    for p in patterns:
        hits += glob.glob(os.path.join(DATA_DIR, p))
    hits = sorted(set(hits))
    if not hits:
        raise FileNotFoundError(f"No file matched: {patterns} in {DATA_DIR}. Found={glob.glob(os.path.join(DATA_DIR,'*.xlsx'))}")
    return hits[0]

EVENT_XLSX   = pick_one(["*Event Module Room*.xlsx"])
ROOMS_XLSX   = pick_one(["*Rooms and Room Types*.xlsx"])
DPT_XLSX     = pick_one(["*DPT Data*.xlsx"])
PC_XLSX      = pick_one(["Programme-Course*.xlsx", "*Programme*Course*.xlsx"])
STUDENT_XLSX = pick_one(["*Student Programme Module Event*.xlsx", "*Student Programme Module*.xlsx"])

print("EVENT_XLSX  :", EVENT_XLSX)
print("ROOMS_XLSX  :", ROOMS_XLSX)
print("DPT_XLSX    :", DPT_XLSX)
print("PC_XLSX     :", PC_XLSX)
print("STUDENT_XLSX:", STUDENT_XLSX)

EVENT_XLSX  : data\2024-5 Event Module Room.xlsx
ROOMS_XLSX  : data\Rooms and Room Types.xlsx
DPT_XLSX    : data\2024-5 DPT Data.xlsx
PC_XLSX     : data\Programme-Course.xlsx
STUDENT_XLSX: data\2024-5 Student Programme Module Event.xlsx


In [5]:
import pandas as pd

events_raw = pd.read_excel(EVENT_XLSX, sheet_name=EVENT_SHEET)
rooms_raw  = pd.read_excel(ROOMS_XLSX, sheet_name=ROOMS_SHEET)
dpt_raw    = pd.read_excel(DPT_XLSX, sheet_name=DPT_SHEET)
pc_raw     = pd.read_excel(PC_XLSX, sheet_name=PC_SHEET)

print("events_raw:", events_raw.shape)
print("rooms_raw :", rooms_raw.shape)
print("dpt_raw   :", dpt_raw.shape)
print("pc_raw    :", pc_raw.shape)

events_raw: (32757, 20)
rooms_raw : (649, 10)
dpt_raw   : (151405, 21)
pc_raw    : (53962, 3)


In [6]:
import re
import pandas as pd
import numpy as np

def snake(s: str) -> str:
    s = str(s).strip().lower()
    s = re.sub(r"[^\w]+", "_", s)
    s = re.sub(r"_+", "_", s).strip("_")
    return s

def parse_timeslot(ts):
    # 'Tuesday 11:00' -> (day, '11:00', minutes)
    if pd.isna(ts):
        return (None, None, None)
    ts = str(ts).strip()
    m = re.match(r"^(Monday|Tuesday|Wednesday|Thursday|Friday|Saturday|Sunday)\s+(\d{1,2}:\d{2})$", ts, re.I)
    if not m:
        return (None, None, None)
    day = m.group(1).capitalize()
    hh, mm = map(int, m.group(2).split(":"))
    return (day, f"{hh:02d}:{mm:02d}", hh*60 + mm)

def parse_weeks(weeks_str):
    if pd.isna(weeks_str):
        return []
    s = str(weeks_str).strip()
    if not s:
        return []
    nums = re.findall(r"\d{1,2}", s)
    return [int(x) for x in nums]

events = events_raw.copy()
rooms  = rooms_raw.copy()
dpt    = dpt_raw.copy()
pc     = pc_raw.copy()

events.columns = [snake(c) for c in events.columns]
rooms.columns  = [snake(c) for c in rooms.columns]
dpt.columns    = [snake(c) for c in dpt.columns]
pc.columns     = [snake(c) for c in pc.columns]

print("events cols:", events.columns.tolist())
print("rooms cols :", rooms.columns.tolist())

events cols: ['module_department', 'module_code', 'module_name', 'event_id', 'event_name', 'event_type', 'duration_minutes', 'event_size', 'timeslot', 'wholeclass', 'online_delivery', 'number_of_weeks', 'weeks', 'room', 'room_type_2', 'room_type_1', 'building', 'campus', 'semester', 'room_lock']
rooms cols : ['id', 'description', 'capacity', 'building', 'building_1', 'campus', 'central_local', 'room_type', 'specialist_room_type', 'has_a_24_5_event']


In [ ]:
for c in ["event_id","module_code","room","timeslot","semester","campus","event_type","weeks",
          "module_department","building","room_lock"]:
    if c in events.columns:
        events[c] = events[c].astype("string").str.strip()

for c in ["duration_minutes","event_size"]:
    if c in events.columns:
        events[c] = pd.to_numeric(events[c], errors="coerce")

if "wholeclass" in events.columns:
    events["wholeclass"] = events["wholeclass"].astype("boolean")

events = events.drop_duplicates()

# baseline day/time
day, start_str, start_min = zip(*events["timeslot"].map(parse_timeslot))
events["baseline_day"] = day
events["baseline_start"] = start_str
events["baseline_start_min"] = start_min

# weeks
if "weeks" in events.columns:
    events["weeks_list"] = events["weeks"].map(parse_weeks)
    events["n_weeks"] = events["weeks_list"].map(len)

print("events cleaned:", events.shape, "unique event_id:", events["event_id"].nunique())
print("timeslot parse fail ratio:", float(events["baseline_day"].isna().mean()))

events cleaned: (32757, 25) unique event_id: 31373
timeslot parse fail ratio: 0.0


In [ ]:
# The rooms table must contain an id field.
if "id" not in rooms.columns:
    raise ValueError(f"Rooms table missing 'id'. Columns={rooms.columns.tolist()}")

rooms["room_id"] = rooms["id"].astype("string").str.strip()
if "capacity" in rooms.columns:
    rooms["capacity"] = pd.to_numeric(rooms["capacity"], errors="coerce")

for c in ["campus","central_local","room_type","specialist_room_type","description","has_a_24_5_event"]:
    if c in rooms.columns:
        rooms[c] = rooms[c].astype("string").str.strip()

rooms = rooms.drop(columns=["id"]).drop_duplicates()

rooms_in_events = set(events["room"].dropna().unique()) if "room" in events.columns else set()
rooms_in_table  = set(rooms["room_id"].dropna().unique())
missing_rooms = sorted(list(rooms_in_events - rooms_in_table))

print("rooms cleaned:", rooms.shape, "unique rooms:", rooms["room_id"].nunique())
print("missing rooms from rooms table:", len(missing_rooms))

if missing_rooms:
    tmp = events.loc[events["room"].isin(missing_rooms)].copy()

    cap_infer = tmp.groupby("room")["event_size"].max().reset_index() \
                   .rename(columns={"room":"room_id","event_size":"capacity"})

    campus_infer = tmp.groupby("room")["campus"].agg(
        lambda x: x.dropna().mode().iloc[0] if len(x.dropna()) else None
    ).reset_index().rename(columns={"room":"room_id"})

    add = cap_infer.merge(campus_infer, on="room_id", how="left")
    add["capacity_source"] = "inferred_from_events"

    base = rooms.assign(capacity_source="rooms_table")
    for c in base.columns:
        if c not in add.columns:
            add[c] = None

    rooms_aug = pd.concat([base, add[base.columns]], ignore_index=True)
else:
    rooms_aug = rooms.assign(capacity_source="rooms_table")

print("rooms_aug:", rooms_aug.shape)

rooms cleaned: (649, 10) unique rooms: 649
missing rooms from rooms table: 48
rooms_aug: (697, 11)


C:\Users\16486\AppData\Local\Temp\ipykernel_84104\1660542202.py:41: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  rooms_aug = pd.concat([base, add[base.columns]], ignore_index=True)


In [9]:
import os
STUDENT_CSV = os.path.join("data", "student_programme_module_event.csv")

In [ ]:
enrol_raw = pd.read_csv(STUDENT_CSV)

enrol = enrol_raw.copy()
enrol.columns = [snake(c) for c in enrol.columns]
for c in enrol.columns:
    enrol[c] = enrol[c].astype("string").str.strip()
enrol = enrol.drop_duplicates()

# Core Aggregation: cohort x event -> number of students
cohort_event_counts = enrol.groupby(["programme_code_year","event_id"]).agg(
    n_students=("anonid","nunique")
).reset_index()

event_att = enrol.groupby("event_id").agg(
    n_students=("anonid","nunique"),
    n_programmes=("programme_code_year","nunique")
).reset_index()

print("enrol:", enrol.shape, "unique students:", enrol["anonid"].nunique())
print("cohort_event_counts:", cohort_event_counts.shape)
print("event_att:", event_att.shape)

enrol: (930173, 8) unique students: 25588
cohort_event_counts: (143573, 3)
event_att: (29105, 3)


In [ ]:
# events + rooms + attendance
events_rows = events.merge(
    rooms_aug[["room_id","capacity","campus","central_local","room_type","specialist_room_type","capacity_source"]],
    left_on="room", right_on="room_id", how="left"
).drop(columns=["room_id"], errors="ignore")

events_rows["baseline_room_missing_in_rooms_table"] = (events_rows["capacity_source"] == "inferred_from_events")
events_rows = events_rows.merge(event_att, on="event_id", how="left")

# event -> candidate rooms (seed)
event_room_options = events_rows[["event_id","room"]].dropna().drop_duplicates().rename(columns={"room":"room_id"})

def first_nonnull(s):
    s = s.dropna()
    return s.iloc[0] if len(s) else None

agg = {k:v for k,v in {
    "module_code": first_nonnull,
    "event_type": first_nonnull,
    "semester": first_nonnull,
    "campus": first_nonnull,
    "wholeclass": "max" if "wholeclass" in events_rows.columns else first_nonnull,
    "duration_minutes": "max" if "duration_minutes" in events_rows.columns else first_nonnull,
    "event_size": "max" if "event_size" in events_rows.columns else first_nonnull,
    "timeslot": first_nonnull,
    "baseline_day": first_nonnull,
    "baseline_start": first_nonnull,
    "baseline_start_min": first_nonnull,
    "room": first_nonnull,
    "capacity": "max" if "capacity" in events_rows.columns else first_nonnull,
    "room_type": first_nonnull,
    "specialist_room_type": first_nonnull,
    "n_students": "max",
    "n_programmes": "max",
}.items() if k in events_rows.columns}

events_master = events_rows.groupby("event_id").agg(agg).reset_index()

print("events_rows:", events_rows.shape)
print("events_master:", events_master.shape)
print("event_room_options:", event_room_options.shape)
print("events_master missing capacity ratio:", float(events_master["capacity"].isna().mean()) if "capacity" in events_master.columns else None)

events_rows: (32757, 34)
events_master: (31373, 17)
event_room_options: (29598, 2)
events_master missing capacity ratio: 0.10072355209893857


In [13]:
OUTDIR = "processed_timetabling"
os.makedirs(OUTDIR, exist_ok=True)

events_master.to_csv(os.path.join(OUTDIR, "events_master.csv"), index=False)
event_room_options.to_csv(os.path.join(OUTDIR, "event_room_options.csv"), index=False)
rooms_aug.to_csv(os.path.join(OUTDIR, "rooms_augmented.csv"), index=False)
cohort_event_counts.to_csv(os.path.join(OUTDIR, "cohort_event_counts.csv"), index=False)

print("Saved core MILP inputs to:", OUTDIR)

Saved core MILP inputs to: processed_timetabling


In [ ]:
import numpy as np

# Source of Missing Statistics
print("missing capacity:", events_master["capacity"].isna().mean())

# Build a capacity_used field after filling
events_master["capacity_used"] = events_master["capacity"]

# Fill with event_size
if "event_size" in events_master.columns:
    events_master.loc[events_master["capacity_used"].isna(), "capacity_used"] = events_master.loc[
        events_master["capacity_used"].isna(), "event_size"
    ]

# Fill in with n_students
events_master.loc[events_master["capacity_used"].isna(), "capacity_used"] = events_master.loc[
    events_master["capacity_used"].isna(), "n_students"
]

global_med = np.nanmedian(pd.to_numeric(events_master["capacity_used"], errors="coerce"))
events_master["capacity_used"] = pd.to_numeric(events_master["capacity_used"], errors="coerce")
events_master["capacity_used"] = events_master["capacity_used"].fillna(global_med)

print("after fill missing capacity_used:", events_master["capacity_used"].isna().mean())
print("global median used for fallback:", global_med)

missing capacity: 0.10072355209893857
after fill missing capacity_used: 0.0
global median used for fallback: 32.0


In [ ]:
# Ensure column names are consistent
cohort_event_counts = cohort_event_counts.rename(columns={
    "programme_code_year": "cohort",
    "event_id": "event_id",
    "n_students": "n_students"
})

print("rows:", len(cohort_event_counts))
print("unique cohorts:", cohort_event_counts["cohort"].nunique())
print("unique events:", cohort_event_counts["event_id"].nunique())

cohort_event_counts.to_csv(
    os.path.join("processed_timetabling", "cohort_clash_input.csv"),
    index=False
)

rows: 143573
unique cohorts: 2091
unique events: 29105


In [16]:
print("avg events per cohort:",
      cohort_event_counts.groupby("cohort")["event_id"].nunique().mean())

print("avg cohorts per event:",
      cohort_event_counts.groupby("event_id")["cohort"].nunique().mean())

avg events per cohort: 68.662362505978
avg cohorts per event: 4.932932485827178


In [ ]:
STEP = 10 

def to_min(t):
    hh, mm = map(int, t.split(":"))
    return 60*hh + mm

START_DAY = to_min("09:00")
END_DAY   = to_min("18:00")

DAYS = ["Monday","Tuesday","Wednesday","Thursday","Friday"]

In [18]:
WINDOW = 120  # ±120分钟

def candidate_starts(row):
    base = row["baseline_start_min"]
    dur  = row["duration_minutes"]
    day  = row["baseline_day"]

    if pd.isna(base) or pd.isna(dur):
        return []

    lower = max(START_DAY, base - WINDOW)
    upper = min(END_DAY - dur, base + WINDOW)

    return list(range(int(lower), int(upper)+1, STEP))

events_master["S0_candidates"] = events_master.apply(candidate_starts, axis=1)
events_master["n_S0"] = events_master["S0_candidates"].map(len)

print(events_master["n_S0"].describe())

count    31373.000000
mean        20.134606
std          5.788195
min          0.000000
25%         16.000000
50%         20.000000
75%         25.000000
max         25.000000
Name: n_S0, dtype: float64


In [19]:
allowed_S0 = (
    events_master[["event_id","baseline_day","S0_candidates"]]
    .explode("S0_candidates")
    .rename(columns={
        "baseline_day":"day",
        "S0_candidates":"start_min"
    })
)

allowed_S0.to_csv(
    os.path.join("processed_timetabling","allowed_start_times_S0.csv"),
    index=False
)

print("S0 total rows:", len(allowed_S0))

S0 total rows: 631979


In [20]:
END_S1 = to_min("17:00")

def filter_S1(row):
    dur = row["duration_minutes"]
    return [s for s in row["S0_candidates"] if s + dur <= END_S1]

events_master["S1_candidates"] = events_master.apply(filter_S1, axis=1)

allowed_S1 = (
    events_master[["event_id","baseline_day","S1_candidates"]]
    .explode("S1_candidates")
    .rename(columns={
        "baseline_day":"day",
        "S1_candidates":"start_min"
    })
)

allowed_S1.to_csv(
    os.path.join("processed_timetabling","allowed_start_times_S1.csv"),
    index=False
)

print("S1 total rows:", len(allowed_S1))

S1 total rows: 570698


In [21]:
FRI_AFTER = to_min("12:00")

def filter_S2(row):
    day = row["baseline_day"]
    dur = row["duration_minutes"]

    out = []
    for s in row["S0_candidates"]:
        if day == "Friday" and s >= FRI_AFTER:
            continue
        if s + dur <= END_DAY:
            out.append(s)
    return out

events_master["S2_candidates"] = events_master.apply(filter_S2, axis=1)

allowed_S2 = (
    events_master[["event_id","baseline_day","S2_candidates"]]
    .explode("S2_candidates")
    .rename(columns={
        "baseline_day":"day",
        "S2_candidates":"start_min"
    })
)

allowed_S2.to_csv(
    os.path.join("processed_timetabling","allowed_start_times_S2.csv"),
    index=False
)

print("S2 total rows:", len(allowed_S2))

S2 total rows: 573071


In [22]:
print("avg S0 per event:", events_master["n_S0"].mean())
print("events with 0 S0:", (events_master["n_S0"]==0).sum())

avg S0 per event: 20.134606190036017
events with 0 S0: 296


In [23]:
zero = events_master.loc[events_master["n_S0"]==0, [
    "event_id","baseline_day","baseline_start_min","duration_minutes","timeslot","semester",
    "wholeclass","room","capacity_used","n_students"
]].copy()

print("zero events:", zero.shape)
zero.head(10)

zero events: (296, 10)


,event_id,baseline_day,baseline_start_min,duration_minutes,timeslot,semester,wholeclass,room,capacity_used,n_students
450,E:1HRAODGOTJ,Saturday,480,945,Saturday 08:00,Semester 2,True,<NA>,42.0,53.0
556,E:1LZA9BQI7U-00002,Thursday,1170,60,Thursday 19:30,Semester 1,False,0551_01_1.26,60.0,NaN
559,E:1LZA9BQI7U-00005,Thursday,1170,60,Thursday 19:30,Semester 1,False,0551_00_G37,25.0,NaN
613,E:1MNQGVGZTW,Thursday,30,1370,Thursday 00:30,Semester 2,True,<NA>,45.0,31.0
614,E:1MNQGVGZTW-00001,Friday,30,1370,Friday 00:30,Semester 2,True,<NA>,45.0,31.0
615,E:1MNQGVGZTW-00002,Saturday,30,1370,Saturday 00:30,Semester 2,True,<NA>,45.0,31.0
616,E:1MNQGVGZTW-00003,Sunday,30,1370,Sunday 00:30,Semester 2,True,<NA>,45.0,31.0
760,E:1TDBWQ2Q3E,Wednesday,1140,90,Wednesday 19:00,Semester 1,False,0564_03_3.15,24.0,NaN
781,E:1UCVCMW7IG,Wednesday,1140,90,Wednesday 19:00,Semester 1,False,0551_-1_LG44,14.0,NaN
1146,E:29CENWDUFB,Thursday,1140,90,Thursday 19:00,Semester 1,False,0551_00_G32,12.0,NaN


In [ ]:
print("missing duration:", zero["duration_minutes"].isna().sum())
print("duration <=0:", (zero["duration_minutes"]<=0).sum())
print("duration > 540:", (zero["duration_minutes"]>540).sum())
zero["duration_minutes"].describe()

print("missing baseline_start_min:", zero["baseline_start_min"].isna().sum())
print(zero["baseline_day"].value_counts(dropna=False).head(10))

missing duration: 0
duration <=0: 0
duration > 540: 56
missing baseline_start_min: 0
baseline_day
Monday       84
Wednesday    64
Tuesday      59
Thursday     55
Friday       12
Saturday     11
Sunday       11
Name: count, dtype: int64


In [25]:
START_DAY = 9*60
END_DAY   = 18*60
WINDOW    = 120

def reason(row):
    day = row["baseline_day"]
    base = row["baseline_start_min"]
    dur = row["duration_minutes"]

    if pd.isna(dur):
        return "missing_duration"
    if dur <= 0:
        return "nonpositive_duration"
    if dur > (END_DAY-START_DAY):
        return "duration_too_long_for_day"
    if day not in ["Monday","Tuesday","Wednesday","Thursday","Friday"]:
        return "non_teaching_day"
    if pd.isna(base):
        return "missing_baseline_start"
    
    lower = max(START_DAY, base - WINDOW)
    upper = min(END_DAY - dur, base + WINDOW)
    if upper < lower:
        return "window_too_tight_near_boundary"
    return "other"

zero["reason"] = zero.apply(reason, axis=1)
zero["reason"].value_counts()

reason
window_too_tight_near_boundary    240
duration_too_long_for_day          56
Name: count, dtype: int64

In [26]:
dur_med_global = events_master["duration_minutes"].median()

if "event_type" in events_master.columns:
    dur_med_by_type = events_master.groupby("event_type")["duration_minutes"].median()
else:
    dur_med_by_type = None

print("global duration median:", dur_med_global)


fix_mask = events_master["duration_minutes"].isna() | (events_master["duration_minutes"]<=0)

if dur_med_by_type is not None and "event_type" in events_master.columns:
    events_master.loc[fix_mask, "duration_minutes"] = events_master.loc[fix_mask, "event_type"].map(dur_med_by_type)

events_master["duration_minutes"] = events_master["duration_minutes"].fillna(dur_med_global)
events_master.loc[events_master["duration_minutes"]<=0, "duration_minutes"] = dur_med_global

global duration median: 80.0


In [ ]:
STEP = 10

def all_day_feasible(dur):
    dur = int(dur)
    return list(range(START_DAY, END_DAY - dur + 1, STEP))

def candidate_starts_with_fallback(row):
    base = row["baseline_start_min"]
    dur  = row["duration_minutes"]
    if pd.isna(dur):
        return []  
    dur = int(dur)

    if pd.isna(base):
        return all_day_feasible(dur)

    lower = max(START_DAY, base - WINDOW)
    upper = min(END_DAY - dur, base + WINDOW)

    if upper < lower:
        return all_day_feasible(dur)   # fallback
    return list(range(int(lower), int(upper)+1, STEP))

events_master["S0_candidates"] = events_master.apply(candidate_starts_with_fallback, axis=1)
events_master["n_S0"] = events_master["S0_candidates"].map(len)

print("avg S0 per event:", events_master["n_S0"].mean())
print("events with 0 S0:", (events_master["n_S0"]==0).sum())

avg S0 per event: 20.454403467950147
events with 0 S0: 56


In [28]:
zero56 = events_master.loc[events_master["n_S0"]==0, [
    "event_id","baseline_day","baseline_start_min","duration_minutes","timeslot","semester",
    "event_type","wholeclass","room","capacity_used","n_students"
]].copy()

print("zero56:", zero56.shape)
print("baseline_day counts:\n", zero56["baseline_day"].value_counts(dropna=False).head(10))
print("duration stats:\n", zero56["duration_minutes"].describe())
print("missing duration:", zero56["duration_minutes"].isna().sum())
print("duration<=0:", (zero56["duration_minutes"]<=0).sum())
print("duration>540:", (zero56["duration_minutes"]>540).sum())

zero56: (56, 11)
baseline_day counts:
 baseline_day
Saturday     11
Sunday       11
Friday       10
Tuesday      10
Thursday      6
Wednesday     6
Monday        2
Name: count, dtype: int64
duration stats:
 count      56.000000
mean     1040.678571
std       385.192425
min       570.000000
25%       600.000000
50%      1370.000000
75%      1370.000000
max      1410.000000
Name: duration_minutes, dtype: float64
missing duration: 0
duration<=0: 0
duration>540: 56


In [ ]:
START_DAY = 9*60
END_DAY   = 18*60
MAX_DAY_DURATION = END_DAY - START_DAY

events_master["is_teaching_day"] = events_master["baseline_day"].isin(["Monday","Tuesday","Wednesday","Thursday","Friday"])
events_master["is_duration_ok"]  = pd.to_numeric(events_master["duration_minutes"], errors="coerce") <= MAX_DAY_DURATION

events_master["is_schedulable_S0"] = events_master["is_teaching_day"] & events_master["is_duration_ok"]

print("schedulable events:", events_master["is_schedulable_S0"].mean(), 
      events_master["is_schedulable_S0"].sum(), "/", len(events_master))

outliers = events_master.loc[~events_master["is_schedulable_S0"],
    ["event_id","baseline_day","timeslot","duration_minutes","event_type","module_code","room","n_students"]
].copy()

outliers.to_csv(os.path.join("processed_timetabling","unschedulable_events_S0.csv"), index=False)
print("unschedulable exported:", outliers.shape)

schedulable events: 0.9971312912376885 31283 / 31373
unschedulable exported: (90, 8)


In [ ]:
sched = events_master.loc[events_master["is_schedulable_S0"]].copy()

WINDOW = 120
STEP = 10
DAYS = ["Monday","Tuesday","Wednesday","Thursday","Friday"]

def s0_pairs(row):
    dur = int(row["duration_minutes"])
    base = row["baseline_start_min"]
    day = row["baseline_day"]

    if pd.isna(base):
        starts = list(range(START_DAY, END_DAY - dur + 1, STEP))
    else:
        lower = max(START_DAY, base - WINDOW)
        upper = min(END_DAY - dur, base + WINDOW)
        if upper < lower:
            starts = list(range(START_DAY, END_DAY - dur + 1, STEP))
        else:
            starts = list(range(int(lower), int(upper)+1, STEP))
    return [(day, s) for s in starts]

sched["S0_pairs"] = sched.apply(s0_pairs, axis=1)
sched["n_S0"] = sched["S0_pairs"].map(len)

print("avg S0 per schedulable event:", sched["n_S0"].mean())
print("events with 0 S0 (schedulable):", (sched["n_S0"]==0).sum())

avg S0 per schedulable event: 20.500271713070997
events with 0 S0 (schedulable): 0


In [31]:
allowed_S0 = sched[["event_id","S0_pairs"]].explode("S0_pairs").dropna()
allowed_S0[["day","start_min"]] = pd.DataFrame(allowed_S0["S0_pairs"].tolist(), index=allowed_S0.index)
allowed_S0 = allowed_S0.drop(columns=["S0_pairs"])

allowed_S0.to_csv(os.path.join("processed_timetabling","allowed_start_times_S0.csv"), index=False)
print("allowed_S0 rows:", len(allowed_S0))

allowed_S0 rows: 641310


In [32]:
cohort_event_counts_final = cohort_event_counts.rename(columns={
    "programme_code_year": "cohort",
    "n_students": "n_students"
})[["cohort","event_id","n_students"]].copy()

cohort_event_counts_final.to_csv(
    os.path.join("processed_timetabling","cohort_clash_input.csv"),
    index=False
)
print("saved cohort_clash_input.csv:", cohort_event_counts_final.shape)

saved cohort_clash_input.csv: (143573, 3)


In [33]:
events_master_sched = events_master.loc[events_master["is_schedulable_S0"]].copy()
events_master_sched.to_csv(
    os.path.join("processed_timetabling","events_master_schedulable.csv"),
    index=False
)
print("saved events_master_schedulable.csv:", events_master_sched.shape)

saved events_master_schedulable.csv: (31283, 25)


In [34]:
allowed_S0.to_csv(os.path.join("processed_timetabling","allowed_start_times_S0.csv"), index=False)
print("saved allowed_start_times_S0.csv:", allowed_S0.shape)

def to_min(t):
    hh, mm = map(int, t.split(":"))
    return 60*hh + mm

END_S1 = to_min("17:00")

tmp = allowed_S0.merge(events_master_sched[["event_id","duration_minutes"]], on="event_id", how="left")
allowed_S1 = tmp.loc[tmp["start_min"] + tmp["duration_minutes"] <= END_S1, ["event_id","day","start_min"]]

allowed_S1.to_csv(os.path.join("processed_timetabling","allowed_start_times_S1.csv"), index=False)
print("saved allowed_start_times_S1.csv:", allowed_S1.shape)

FRI_AFTER = to_min("12:00")

allowed_S2 = allowed_S0.loc[
    ~((allowed_S0["day"]=="Friday") & (allowed_S0["start_min"]>=FRI_AFTER)),
    ["event_id","day","start_min"]
]

allowed_S2.to_csv(os.path.join("processed_timetabling","allowed_start_times_S2.csv"), index=False)
print("saved allowed_start_times_S2.csv:", allowed_S2.shape)

saved allowed_start_times_S0.csv: (641310, 3)
saved allowed_start_times_S1.csv: (578472, 3)
saved allowed_start_times_S2.csv: (580554, 3)


In [35]:
event_room_options.to_csv(os.path.join("processed_timetabling","event_room_options.csv"), index=False)
rooms_aug.to_csv(os.path.join("processed_timetabling","rooms_augmented.csv"), index=False)

print("saved event_room_options.csv:", event_room_options.shape)
print("saved rooms_augmented.csv:", rooms_aug.shape)

saved event_room_options.csv: (29598, 2)
saved rooms_augmented.csv: (697, 11)


In [36]:
uns = outliers.copy()

def uns_reason(row):
    if row["baseline_day"] not in ["Monday","Tuesday","Wednesday","Thursday","Friday"]:
        return "non_teaching_day"
    if row["duration_minutes"] > 540:
        return "duration_exceeds_teaching_day"
    return "other"

uns["reason"] = uns.apply(uns_reason, axis=1)

uns.to_csv(os.path.join("processed_timetabling","unschedulable_events_S0.csv"), index=False)
print("saved unschedulable_events_S0.csv:", uns.shape)
print(uns["reason"].value_counts())

saved unschedulable_events_S0.csv: (90, 9)
reason
non_teaching_day                 56
duration_exceeds_teaching_day    34
Name: count, dtype: int64
